<table>

<thead >
<tr>
<th>
<p>Syntax to pass to the .select() method</p>
</th>
<th>
<p>Match Results</p>
</th>
</tr>
</thead>
<tbody>
<tr>
<td>
<p><code>soup.select('div')</code></p>
</td>
<td>
<p>All elements with the <code>&lt;div&gt;</code> tag</p>
</td>
</tr>
<tr>
<td>
<p><code>soup.select('#some_id')</code></p>
</td>
<td>
<p>The HTML element containing the <code>id</code> attribute of <code>some_id</code></p>
</td>
</tr>
<tr>
<td>
<p><code>soup.select('.notice')</code></p>
</td>
<td>
<p>All the HTML elements with the CSS <code>class</code> named <code>notice</code></p>
</td>
</tr>
<tr>
<td>
<p><code>soup.select('div span')</code></p>
</td>
<td>
<p>Any elements named <code>&lt;span&gt;</code> that are within an element named <code>&lt;div&gt;</code></p>
</td>
</tr>
<tr>
<td>
<p><code>soup.select('div &gt; span')</code></p>
</td>
<td>
<p>Any elements named <code class="literal2">&lt;span&gt;</code> that are <span><em >directly</em></span> within an element named <code class="literal2">&lt;div&gt;</code>, with no other element in between</p>
</td>
</tr>
<tr>

</tr>
</tbody>
</table>

In [3]:
import requests 
import bs4

### User-Agent ne işe yarar?
HTTP isteklerinde sunucuya kendini tanıtan bir kimlik bilgisidir. Sunucu, gelen isteğin bir Chrome tarayıcısından mı, telefondan mı yoksa bir Python script'inden mi geldiğini bu başlığa bakarak anlar.

* Varsayılan `requests.get()` isteğinde bu başlık `python-requests/2.x.x` olarak gider. Birçok site (Wikipedia dahil) bu başlıklı bot isteklerini doğrudan bloklar (genellikle `403 Forbidden` döner).

### O bilgiler rastgele mi yazıldı?
Kişisel bilgisayarın bilgilerini içeren bir metin değildir; standart bir macOS/Chrome tarayıcısının kullandığı kalıp bir User-Agent dizesidir. Sen kod içine ne yazarsan karşı sunucuya o iletilir. İsteği gerçek bir tarayıcı yapıyormuş gibi göstermek için yaygın bir tarayıcı formatı kullanılmıştır.

In [12]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

In [13]:
res = requests.get('https://en.wikipedia.org/wiki/Grace_Hopper',headers=headers)

In [15]:
soup = bs4.BeautifulSoup(res.text, "lxml")
soup

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-feature-navigation-update-disabled vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="en">
<head>
<meta charset="utf-8"/>
<title>Grace Hopper - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vecto

In [20]:
# normalde soup.select(".vector-toc-text")  yazacaktık ama yeni değişikliklerde <div>'i ve içindeki tüm çocuk etiketleri bir bütün olarak çeker

#### Neden numaralar (`vector-toc-numb`) ve fazladan etiketler geldi?

* **HTML Yapısı Değişti:** Videonun çekildiği eski Wikipedia tasarımında başlık metni doğrudan `<span class="toctext">Career</span>` içindeydi; başlık numaraları o etiketin dışındaydı.
* **Yeni "Vector" Teması:** Yeni yapıda `vector-toc-text` sınıfı en dıştaki `<div>` kapsayıcısıdır. Bu kapsayıcının içine hem numara (`<span class="vector-toc-numb">2</span>`) hem de başlık metni (`<span>Career</span>`) yerleştirilmiştir.
* Sen `.vector-toc-text` seçtiğinde BeautifulSoup o `<div>`'i ve içindeki **tüm çocuk etiketleri** bir bütün olarak çeker.

#### Sadece başlık yazılarını almak için:

```python
for item in soup.select('.vector-toc-text'):
    print(item.text)

ama bu da bize yetmiyor çünkü numaraları vs de gözüküyor, örnek:

(Top)

1
Early life and education


2
Career


2.1
World War II

In [35]:
for item in soup.select('.vector-toc-text'):
    spans = item.find_all('span')
    if spans:
        print(spans[-1].text)

Early life and education
Career
World War II
UNIVAC
COBOL
Standards
Retirement
Post-retirement
Anecdotes
Death
Dates of rank
Awards and honors
Military awards
Other awards
Legacy
Places
Programs
In popular culture
Grace Hopper Celebration of Women in Computing
See also
Notes
References
Obituary notices
Further reading
External links


### Nasıl çalışır?

* `(Top)` başlığında `span` olmadığı için otomatik olarak atlanır.
* Numaralı başlıklarda ilk `span` numarayı (`1`, `2.1`), son `span` ise başlık ismini (`Early life...`, `UNIVAC`) tuttuğu için `spans[-1]` doğrudan saf başlık metnini yazdırır.

In [33]:
soup.select(".vector-toc-text")[1].getText()

'\n1\nEarly life and education\n'

In [23]:
# bu da eski yazım, bahsettiğim div ve içindeki tüm çocuk etiketleri de çekiyor
soup.select(".vector-toc-text") # . css de class ifade eder '#' ise ID o yüzden başına . koyduk

[<div class="vector-toc-text">(Top)</div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">1</span>
 <span>Early life and education</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">2</span>
 <span>Career</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">2.1</span>
 <span>World War II</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">2.2</span>
 <span>UNIVAC</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">2.3</span>
 <span>COBOL</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">2.4</span>
 <span>Standards</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">3</span>
 <span>Retirement</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">4</span>
 <span>Post-retirement</span>
 </div>,
 <div class="vector-toc-text">
 <span class="vector-toc-numb">5</span>
 <span>Anecdotes</span>
 </div>